In [1]:
from icecream import ic

## Carrega modelos

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

In [3]:
def load_llm(model_name):
    
    # load the tokenizer and the model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype="auto",
        device_map="auto"
    )

    return model, tokenizer


def load_vectorstore(model_name="Qwen/Qwen3-Embedding-0.6B", device='cpu'):

    encoder = SentenceTransformer(model_name, device=device)
    client = QdrantClient(path="../metadados/VectorStore/") # Carrega vectorstore em disco

    return encoder, client

## Logica de agentes

### Estado do agente (Agent state)

In [5]:
from typing import Dict, TypedDict

In [ ]:
class AgentState(TypedDict):
    client: any
    encoder: any
    
    query: str
    catalogs: dict

    messages: str
    #========= temp ==========
    #========= LLM model =====
    tokenizer: any
    model: any

    model_config: str


### Tools

In [ ]:
from langchain_core.tools import tool # Documenta tools de forma apropriada para ser enviada a llm

#### LLM CALL


In [ ]:
import torch
import gc

def call_llm(state: AgentState) -> AgentState:

    tokenizer = state['tokenizer']
    messages = state['messages']
    

    text = tokenizer.apply_chat_template( # Formata 
        messages,
        tokenize=False,
        **self.model_config[self.model.config.name_or_path]['template']
    )
    #print(text)
    model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

    # conduct text completion
    with torch.no_grad():
        generated_ids = self.model.generate(
            **model_inputs,
            **self.model_config[self.model.config.name_or_path]['generate']
        )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = self.tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = self.tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    
    #print(f"thinking content: \n{thinking_content}\n")
    #print(f"ACTION: \n{content}\n")

    gc.collect()
    torch.cuda.empty_cache()

    return content, thinking_content

#### OpenDataSearch

In [7]:
def open_data_search(state: AgentState) -> AgentState:
    
    '''

        Nome: open_data_search
        Descrição: Busca conjuntos de dados com base em palavras-chave.

        Argumentos:

            {
            "thought": "Breve raciocínio sobre o que fazer a seguir",
            "tool_name": "open_data_search",
            "parameters": {
                "query": "value"
                }
            }

        Retorno:

            {id :

                {
                "score:": score,
                "id": id,
                "Titulo: ": title),
                "Nome: ": nome,
                "Descrição: ": descricao,
                }
            }

'''
    #translator = Translator()

    client = state['client']
    encoder = state['encoder']

    task = "Você é um motor de busca, devolva dados mais relevantes com base na busca"

    query_pt = f"Instrução: {task} Query: {state['query']}"
    query = query_pt
    #query_en = await translator.translate(query_pt, src='pt', dest='en')

    #query = query_en.text

    hits = client.query_points(
        collection_name="Catalogo_metadados",
        query=encoder.encode(query).tolist(),
        limit=5,
    ).points

    catalogs = {} # catalogo temporario retornado, respectivo a cada query individual

    for hit in hits:
        id = hit.payload.get('id')
        if id not in catalogs.keys(): # garante que não há catalogos repitidos
            catalogs.update({ id :
                {"score:": hit.score,
                "id": id,
                "Titulo: ": hit.payload.get('title'),
                "Nome: ": hit.payload.get('nome'),
                "Descrição: ": hit.payload.get('descricao'),
                #"Nome organização": hit.payload.get('nomeOrganizacao'),
                #"catalogacao": hit.payload.get('catalogacao'),
                #"ultimaAtualizacaoDados'": hit.payload.get('ultimaAtualizacaoDados'), 
                }
            })
    #state['catalogs'].update(catalogs)
    ic(catalogs)
    return state


#### ConsultCatalogs

In [8]:
def consult_catalogs(state: AgentState) -> AgentState:
    '''
        tool_name = "consult_catalogs"
        
    '''

    print("Consultando catalogos")

    return state

#### documentReader

In [9]:
import gc
import torch
import gc

def clear_cuda():
    gc.collect()                 # Coleta lixo do Python (variáveis sem referência)
    torch.cuda.empty_cache()     # Esvazia o cache de memória alocada pelo PyTorch
    torch.cuda.ipc_collect()

def batch_encodding(batch: list, encoder) -> list:
    return encoder.encode(batch)

clear_cuda()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid
from pathlib import Path

def external_doc(state: AgentState) -> AgentState:

    client = state['client']
    encoder = state['encoder']

    file_path = "/home/migueldcarvalho/Downloads/edital_petrobras.pdf"

    # 1. Load the PDF (This creates one Document object per page)
    loader = PyPDFLoader(file_path)
    docs = loader.load()

    # 2. Initialize Splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = text_splitter.split_documents(docs)

    print(f"Generated {len(chunks)} chunks.")
    # This chunk will retain metadata like {'source': '...', 'page': 1}
    print(chunks[0].metadata)

    texts = [chunk.page_content for chunk in chunks] # extrai somente texto, sem metadata 

    print("Generating embeddings...")

    batch_size = 20
    vectors = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i+batch_size]
        ic(i, batch)
        vectors.append(batch_encodding(batch, encoder))
        del batch

    collection_name = Path(file_path).name

    client.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(),
        distance=models.Distance.COSINE
        )
    )
    points = []
    for i, chunk in enumerate(chunks):
        print(i)
        # Cria estrutura de Point do qdrant
        points.append(models.PointStruct(
            id=str(uuid.uuid4()),
            vector=vectors[i].tolist(),
            payload={
                "page_content": chunk.page_content, # conteudo
                "metadata": chunk.metadata,  # metadata (paginas...)
                "chunk_index": i
            }
        ))

    operation_info = client.upsert( # Adciona ou atualiza Points
        collection_name=collection_name,
        points=points
    )
state1 = AgentState(client=client, encoder=encoder)
external_doc(state1)

Generated 236 chunks.
{'producer': 'Microsoft® Word para Microsoft 365', 'creator': 'Microsoft® Word para Microsoft 365', 'creationdate': '2023-12-22T17:06:37-03:00', 'msip_label_98fb4d56-f1b0-4a29-9cca-d42fddf5cd3f_enabled': 'true', 'msip_label_98fb4d56-f1b0-4a29-9cca-d42fddf5cd3f_setdate': '2023-12-06T17:58:23Z', 'msip_label_98fb4d56-f1b0-4a29-9cca-d42fddf5cd3f_method': 'Privileged', 'msip_label_98fb4d56-f1b0-4a29-9cca-d42fddf5cd3f_name': 'CONFIDENCIAL_SUBLABEL-2', 'msip_label_98fb4d56-f1b0-4a29-9cca-d42fddf5cd3f_siteid': '5b6f6241-9a57-4be4-8e50-1dfa72e79a57', 'msip_label_98fb4d56-f1b0-4a29-9cca-d42fddf5cd3f_actionid': 'f090123a-3f4f-4660-9dff-aeb420811dc7', 'msip_label_98fb4d56-f1b0-4a29-9cca-d42fddf5cd3f_contentbits': '2', 'author': 'Rejane Ribeiro de Oliveira', 'moddate': '2023-12-22T17:06:37-03:00', 'source': '/home/migueldcarvalho/Downloads/edital_petrobras.pdf', 'total_pages': 63, 'page': 0, 'page_label': '1'}
Generating embeddings...


ic| i: 0
    batch: ['PETRÓLEO BRASILEIRO S.A. – PETROBRAS 
           '
            'PROCESSO SELETIVO PÚBLICO PARA PREENCHIMENTO DE VAGAS E FORMAÇÃO DE '
            'CADASTRO 
           '
            'NO CARGO DE PROFISSIONAL PETROBRAS DE NÍVEL TÉCNICO JÚNIOR 
           '
            'EDITAL Nº 1 – PETROBRAS/PSP RH 2023.2 
           '
            'PETRÓLEO BRASILEIRO S.A. (PETROBRAS) realizará processo seletivo público '
            'para provimento de 
           '
            'vagas e formação de cadastro, mediante condições estabelecidas neste '
            'edital. 
           '
            ' 
           '
            '1 DAS DISPOSIÇÕES PRELIMINARES 
           '
            ' 
           '
            '1.1 O processo seletivo público PETROBRAS/PSP RH 2023.2 será regido por este '
            'edital e executado 
           '
            'pelo Centro Brasileiro de Pesquisa em Avaliação e Seleção e de Promoção de '
            'Eventos (Cebraspe).  
           '
            '1

OutOfMemoryError: CUDA out of memory. Tried to allocate 100.00 MiB. GPU 0 has a total capacity of 5.64 GiB of which 80.69 MiB is free. Including non-PyTorch memory, this process has 5.54 GiB memory in use. Of the allocated memory 5.21 GiB is allocated by PyTorch, and 235.80 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Carregando Modelo

In [ ]:
model, tokenizer = load_llm("Qwen/Qwen3-0.6B") # Modelo base, carregado do HF
#encoder, client = load_vectorstore(device='cuda')

In [ ]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.tools import tool

llm_wrapper = HuggingFacePipeline( # Wrapper do langchain, permite conectar o modelo com outras funcionalidades do langchain como ChatHuggingFace
    model=model, # Modelo carreagado da biblioteca transformers
    tokenizer=tokenizer, # Tokenizer carreagado da biblioteca transformers

    pipeline_kwargs={
        "max_new_tokens": 2048,
        "return_full_text": False, # Importante para evitar repetição do prompt
        "temperature": 0.3,
    }
)

'''
A classe ChatHuggingFace facilita controle do histórico de conversas gerenciando AIMessages, ToolsMessages, HumanMessages, SystemMessages.

Decorator @tool: Transforma funções em objetos da classe StructuredTool, assim podendo ser lidos pelo método bind_tools, informando argumentos de entrada e a saída da função,
descrição da função(DocString) informada no ato de criação da função.

Método bind_tools: Recebe funções sobre o decorator @tool a fim de transformar a função em uma tool para a llm. Esse método lerá o objeto criado pelo
decorator @tool para passar a LLM informção relevantes como descrição, informações de entrada e saída  em formato JSON da tool no momento da inferência. 

Será passado ao modelo uma entrada estruturada em formato json informando descrição, entrada e saída das tools disponíveis.
'''

# Wrapper para o modelo servir como chat, gerencia messagens como AIMessages, ToolsMessages, HumanMessages, SystemMessages
# Formata a entrada para modelo de forma a disponibilizar tools adequadamente utillizando chat_template (tokenizer.apply_chat_template())
# Utiliza o método bind_tools para carregar documentação de tools  

chat_model = ChatHuggingFace(llm=llm_wrapper) # Objeto base, não munido de tools disponíveis

# Agente munido das tools disponíveis, o método bind_tools pass as informações de cada tool para o modelo no momento de inference
#Agent = chat_model.bind_tools(tools)

In [2]:
from langchain_core.utils.function_calling import convert_to_openai_tool
import json

In [4]:
from langchain_core.tools import tool

@tool
def add(a: float, b:float) -> float:
    '''soma dois numeros do tipo float'''
    return a + b 

In [5]:
schema_bruto = convert_to_openai_tool(add)
print("\n--- O JSON QUE VAI PRO LLM ---")
print(json.dumps(schema_bruto, indent=2))


--- O JSON QUE VAI PRO LLM ---
{
  "type": "function",
  "function": {
    "name": "add",
    "description": "soma dois numeros do tipo float",
    "parameters": {
      "properties": {
        "a": {
          "type": "number"
        },
        "b": {
          "type": "number"
        }
      },
      "required": [
        "a",
        "b"
      ],
      "type": "object"
    }
  }
}


### Criando grafo

In [2]:
from langgraph.graph import StateGraph, START, END
import utils

In [ ]:
graph = StateGraph(AgentState)



app = graph.compile()

In [ ]:
model_config = utils.read_yaml(path="../config/model_config")

query = "infecções"
result = app.invoke({'client': client, 'encoder': encoder})


ic| catalogs: {'0364f1c3-6198-4a17-982e-1d325ca8d8fd': {'Descrição: ': 'Taxa de infecção '
                                                                       'hospitalar.',
                                                        'Nome: ': '04-infeccao-hospitalar',
                                                        'Titulo: ': '04. Infecção Hospitalar',
                                                        'id': '0364f1c3-6198-4a17-982e-1d325ca8d8fd',
                                                        'score:': 0.6191945931294637},
               '5a8c36dc-65fd-4159-b3d7-ee0d5350c874': {'Descrição: ': 'Taxa de infecção nas '
                                                                       'unidades de '
                                                                       'internação',
                                                        'Nome: ': 'infeccao-hospitalar-huwc',
                                                        'Titulo: ': 'Infecção Hospitala